## Self Attention 第一重

In [1]:
import math
import torch
import torch.nn as nn

d:\anaconda3\envs\tf2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
class Self_Attention_v1(nn.Module):
    def __init__(self, dim: int=512):
        super().__init__()

        self.dim = dim
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        
    def forward(self, X):
        q = self.q_proj(X) 
        k = self.k_proj(X)
        v = self.v_proj(X)

        # qkv shape: (batch, seq, dim) = X shape
        attention_value = torch.matmul(
            q, k.transpose(-1, -2)
        )
        attention_weight = torch.softmax(
            attention_value / math.sqrt(self.dim),
            dim=-1
        )
        print(f'attention_weight:{attention_weight}')
        output = torch.matmul(attention_weight, v)

        return output


X = torch.rand(3, 4, 6)
SA1 = Self_Attention_v1(6)
output = SA1(X)
output

attention_weight:tensor([[[0.2382, 0.2757, 0.2252, 0.2608],
         [0.2480, 0.2619, 0.2362, 0.2539],
         [0.2447, 0.2659, 0.2328, 0.2565],
         [0.2402, 0.2717, 0.2302, 0.2579]],

        [[0.2362, 0.2680, 0.2372, 0.2587],
         [0.2339, 0.2676, 0.2406, 0.2578],
         [0.2331, 0.2684, 0.2367, 0.2618],
         [0.2342, 0.2702, 0.2370, 0.2587]],

        [[0.2620, 0.2338, 0.2907, 0.2135],
         [0.2600, 0.2406, 0.2733, 0.2261],
         [0.2624, 0.2348, 0.2863, 0.2165],
         [0.2585, 0.2405, 0.2699, 0.2310]]], grad_fn=<SoftmaxBackward0>)


tensor([[[ 0.0072,  0.0251, -0.1880,  0.1641, -0.0393, -0.3597],
         [ 0.0062,  0.0187, -0.1890,  0.1654, -0.0432, -0.3560],
         [ 0.0065,  0.0206, -0.1887,  0.1650, -0.0419, -0.3571],
         [ 0.0073,  0.0232, -0.1884,  0.1647, -0.0409, -0.3582]],

        [[ 0.1134,  0.1367, -0.1363,  0.2814,  0.0948, -0.3732],
         [ 0.1136,  0.1363, -0.1365,  0.2809,  0.0939, -0.3730],
         [ 0.1150,  0.1373, -0.1361,  0.2821,  0.0959, -0.3728],
         [ 0.1141,  0.1373, -0.1366,  0.2813,  0.0949, -0.3727]],

        [[ 0.1488,  0.1452, -0.0140,  0.2368,  0.0466, -0.5085],
         [ 0.1439,  0.1419, -0.0186,  0.2367,  0.0413, -0.5096],
         [ 0.1479,  0.1443, -0.0149,  0.2369,  0.0454, -0.5087],
         [ 0.1429,  0.1408, -0.0201,  0.2365,  0.0401, -0.5103]]],
       grad_fn=<UnsafeViewBackward0>)

## Self Attention 第二重

In [8]:
class Self_Attention_v2(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.dim = dim
        self.proj = nn.Linear(dim, dim * 3)

    def forward(self, X):
        q, k, v = torch.split(self.proj(X),
                            self.dim,
                            dim=-1)
        attention_weight = torch.softmax(
            torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.dim),
            dim=-1
        )
        print(f'attention_weight:{attention_weight}')
        output = attention_weight @ v
        return output
X = torch.rand(2, 4, 5)
SA2 = Self_Attention_v2(5)
output = SA2(X)
print(output)

attention_weight:tensor([[[0.2449, 0.2506, 0.2484, 0.2561],
         [0.2687, 0.2305, 0.2493, 0.2516],
         [0.2660, 0.2251, 0.2509, 0.2580],
         [0.2630, 0.2225, 0.2537, 0.2608]],

        [[0.2665, 0.2527, 0.2504, 0.2303],
         [0.2589, 0.2504, 0.2459, 0.2449],
         [0.2779, 0.2554, 0.2391, 0.2277],
         [0.2527, 0.2435, 0.2618, 0.2420]]], grad_fn=<SoftmaxBackward0>)
tensor([[[ 0.3256, -0.0704, -0.1727,  0.2484, -0.0661],
         [ 0.3338, -0.0643, -0.1752,  0.2592, -0.0778],
         [ 0.3344, -0.0641, -0.1750,  0.2613, -0.0765],
         [ 0.3345, -0.0643, -0.1748,  0.2620, -0.0750]],

        [[ 0.2780, -0.0329, -0.1903,  0.2835, -0.3054],
         [ 0.2740, -0.0326, -0.1885,  0.2804, -0.3039],
         [ 0.2803, -0.0299, -0.1923,  0.2892, -0.3114],
         [ 0.2727, -0.0372, -0.1873,  0.2744, -0.2982]]],
       grad_fn=<UnsafeViewBackward0>)


## Self Attention 第三重

In [ ]:
class Self_Attention_v3(nn.Module):
    def __init__(self, dim: int=512, dropout_rate: float=0.1):
        super().__init__()

        self.dim = dim
        self.dropout = nn.Dropout(dropout_rate)
        self.proj = nn.Linear(dim, dim * 3)
        self.output_proj = nn.Linear(dim, dim)

    def forward(self, X, attention_mask=None):
        qkv = self.proj(X)
        q, k, v = torch.split(
            qkv,
            self.dim,
            dim=-1
        )
        attention_value = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.dim)
        if attention_mask is not None:
            attention_weight = attention_value.masked_fill(
                attention_mask == 0,
                float('-1e20')
            )
        attention_weight = torch.softmax(attention_weight, dim=-1)
        attention_weight = self.dropout(attention_weight)
        result = attention_weight @ v
        output = self.output_proj(result)
        return output



## Self Attention 第四重

In [7]:
import torch
import torch.nn as nn
import math

class SelfAttentionV4(nn.Module):
    def __init__(self, dim: int, dropout_rate: float=0.1):
        super().__init__()

        self.dim = dim
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout_rate)
    
    def forward(self, X, attention_mask=None):
        q = self.q_proj(X)
        k = self.k_proj(X)
        v = self.v_proj(X)

        attention_weight = q @ k.transpose(-1, -2) / math.sqrt(self.dim)
        
        if attention_mask is not None:
            attention_weight.masked_fill(
                attention_mask == 0,
                float('-inf')
            )
        attention_weight = torch.softmax(attention_weight, dim=-1)
        attention_weight = self.dropout(attention_weight)
        result = attention_weight @ v
        output = self.out_proj(result)
        return output
    
X = torch.rand(3, 4, 2)

attention_mask = torch.tensor(
    [
    [1, 1, 1, 0],
    [1, 1, 0, 0],
    [1, 0, 0, 0]
]
)

print(attention_mask.size())
attention_mask = attention_mask.unsqueeze(dim=1).repeat(1, 4, 1)
print(attention_mask.size())
net = SelfAttentionV4(2)
output = net(X, attention_mask=attention_mask)
print(output)

torch.Size([3, 4])
torch.Size([3, 4, 4])
tensor([[[0.3322, 0.5131],
         [0.3323, 0.5130],
         [0.3324, 0.5129],
         [0.3326, 0.5127]],

        [[0.4134, 0.4183],
         [0.4148, 0.4169],
         [0.4144, 0.4172],
         [0.4119, 0.4197]],

        [[0.3052, 0.5359],
         [0.3033, 0.5378],
         [0.3642, 0.4665],
         [0.3161, 0.5080]]], grad_fn=<ViewBackward0>)


## 多头注意力机制

In [15]:
import math
import torch
import torch.nn as nn

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim, head_num, dropout_rate: float=0.1):
        super().__init__()

        self.dim = dim
        self.head_num = head_num
        self.head_dim = self.dim // self.head_num
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.drouput = nn.Dropout(dropout_rate)
    
    def forward(self, X, attention_mask=None):
        batch, seq_len, _ = X.shape
        # q, k, v shape is: (batch, seq_len, dim)
        q = self.q_proj(X)
        k = self.k_proj(X)
        v = self.v_proj(X)

        #  (batch, head_num, seq_len, head_dim)
        q_state = q.view(batch, seq_len, self.head_num, self.head_dim).transpose(1, 2)
        k_state = k.view(batch, seq_len, self.head_num, self.head_dim).transpose(1, 2)
        v_state = v.view(batch, seq_len, self.head_num, self.head_dim).transpose(1, 2)

        attention_weight = torch.matmul(q_state, k_state.transpose(-1, -2)) / math.sqrt(self.head_dim)
        if attention_mask is not None:
            attention_weight.masked_fill(
                attention_mask == 0,
                float('-inf')
            )
        attention_weight = torch.softmax(attention_weight, dim=-1)  # (batch, head_num, seq_len, seq_len)  (3, 8, 2, 2)
        attention_weight = self.drouput(attention_weight)
        output_mid = torch.matmul(attention_weight, v_state)    # (batch, head_num, seq_len, head_dim)
        output = output_mid.transpose(1, 2).contiguous().view(batch, seq_len, -1)
        output = self.out_proj(output)
        return output
    
attention_mask = (
    torch.tensor(
        [
            [0, 1],
            [0, 0],
            [1, 0],
        ]
    )
    .unsqueeze(1)      # 在第1维插入维度 → shape: (3, 1, 2)
    .unsqueeze(2)      # 在第2维插入维度 → shape: (3, 1, 1, 2)
    .expand(3, 8, 2, 2) # 广播扩展 → final shape: (3, 8, 2, 2)
)
X = torch.rand(3, 2, 128)
net = MultiHeadSelfAttention(128, head_num=8)
output = net(X, attention_mask=attention_mask)
print(output)


tensor([[[-3.6042e-02, -1.3670e-01,  5.0720e-01,  4.3111e-02,  4.9796e-02,
          -9.7831e-02, -4.0590e-01, -3.1112e-01, -7.1167e-01, -1.3318e-01,
           1.4363e-01, -2.1665e-01, -1.2350e-01, -1.5196e-01,  2.0307e-01,
           5.7574e-02, -4.5151e-02,  2.6691e-01,  5.2369e-02, -2.0979e-01,
           1.6860e-01,  1.2277e-01, -2.3321e-01,  2.0274e-01,  1.7532e-01,
          -5.5455e-03, -5.4210e-02, -4.1018e-02, -1.4019e-01, -2.0961e-01,
           1.8065e-01, -4.5323e-02, -2.0833e-01, -1.9810e-01,  2.6869e-01,
          -3.5570e-03, -1.8806e-01, -2.3698e-01, -3.0639e-01, -1.9393e-01,
          -2.8078e-01,  1.8515e-01,  2.2330e-02,  1.4450e-01,  7.3664e-02,
          -2.2663e-02,  4.6784e-01, -1.3808e-01,  4.4578e-03,  3.2064e-01,
          -5.1218e-02, -3.0711e-02,  1.0523e-01, -1.8947e-01, -2.4196e-01,
          -1.2058e-02, -1.3066e-01,  1.4319e-01, -9.9202e-02,  1.2707e-01,
          -4.6429e-02, -3.4759e-01,  1.1010e-01,  3.1656e-01,  8.1165e-02,
          -1.4082e-02, -1

In [10]:
X = torch.rand(3, 4, 2)

attention_mask = torch.tensor(
    [
    [1, 1, 1, 0],
    [1, 1, 0, 0],
    [1, 0, 0, 0]
]
)

print(attention_mask.size())
attention_mask = attention_mask.unsqueeze(dim=1)
print(attention_mask.size())

torch.Size([3, 4])
torch.Size([3, 1, 4])


In [3]:
import torch
a = torch.tril(torch.ones(4, 4))
a


tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])